,amount_ratio,new_payee,hour
count,20000.000000,20000.000000,20000.00000
mean,1.329764,0.148850,13.92675
std,1.440717,0.355949,5.59055
min,0.100882,0.000000,0.00000
25%,0.645351,0.000000,11.00000
50%,1.000000,0.000000,16.00000
75%,1.557651,0.000000,18.00000
max,31.508327,1.000000,21.00000


In [2]:
import pandas as pd, numpy as np
REPO = "/home/ibm-user/shared/safecircle"

COLS = {"user": "CONT_ID", "amount": "TX_TOTAL_AMT", "merchant": "TX_MRCHNT_ID",
        "time": "DATE_CREDIT", "fraud": "FRAUD_VER_RESULT"}

df = pd.read_csv("/home/ibm-user/transactions.csv", nrows=100_000)
df = df.dropna(subset=[COLS["user"], COLS["amount"]])
df["user_id"] = df[COLS["user"]]
df["amount"] = df[COLS["amount"]].astype(float)
df = df[df["amount"] > 0].copy()

df["hour"] = df[COLS["time"]].astype(str).str[11:13].astype(int)
df = df.sort_values(["user_id", COLS["time"]])

usual = df.groupby("user_id")["amount"].transform("median")
df["amount_ratio"] = (df["amount"] / usual).clip(upper=100)
df["new_payee"] = (df.groupby(["user_id", COLS["merchant"]]).cumcount() == 0).astype(int)

FEATURES = ["amount_ratio", "new_payee", "hour"]
X = df[FEATURES]
X.describe()

,amount_ratio,new_payee,hour
count,20000.000000,20000.000000,20000.00000
mean,1.329764,0.148850,13.92675
std,1.440717,0.355949,5.59055
min,0.100882,0.000000,0.00000
25%,0.645351,0.000000,11.00000
50%,1.000000,0.000000,16.00000
75%,1.557651,0.000000,18.00000
max,31.508327,1.000000,21.00000


In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
import joblib

model = IsolationForest(n_estimators=100, random_state=42, n_jobs=1)
model.fit(X)

anomaly = -model.score_samples(X)
p50, p99 = np.percentile(anomaly, [50, 99])

y = df[COLS["fraud"]].astype(str).str.strip().str.lower().isin(["y", "yes", "1", "true"]).astype(int).values
print("Fraud cases in sample:", y.sum(), "of", len(y))
if y.sum() > 0:
    print("AUC:", round(roc_auc_score(y, anomaly), 3))
    top = anomaly >= np.percentile(anomaly, 99)
    print(f"Fraud rate overall: {y.mean():.3%} | in the 1% most unusual: {y[top].mean():.3%}")

joblib.dump({"model": model, "features": FEATURES, "p50": p50, "p99": p99},
            f"{REPO}/models/transaction_model.joblib")
print("saved")

Fraud cases in sample: 2 of 20000
AUC: 0.989
Fraud rate overall: 0.010% | in the 1% most unusual: 0.000%
saved


In [4]:
%%writefile /home/ibm-user/shared/safecircle/src/transactions.py
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

REPO = Path(__file__).resolve().parent.parent
MODEL_PATH = REPO / "models" / "transaction_model.joblib"
HISTORY_PATH = REPO / "scenarios" / "history.csv"

RISKY_CATEGORIES = {"gift_card", "wire", "money_transfer"}
RISKY_REASON = "gift-card or wire payment"   # risk.py looks for this exact text

_model = None
_history = None

def _load():
    global _model, _history
    if _history is None:
        _history = pd.read_csv(HISTORY_PATH)
    if _model is None and MODEL_PATH.exists():
        _model = joblib.load(MODEL_PATH)

def _features(txn):
    amount = float(txn["amount"])
    past = _history[_history["user_id"] == txn["user_id"]]
    usual = past["amount"].median() if len(past) else 50.0
    return {
        "amount_ratio": min(amount / usual, 100.0) if usual > 0 else 1.0,
        "new_payee": int(txn.get("is_new_payee", 0)),
        "risky_category": int(str(txn.get("merchant_category", "")).lower() in RISKY_CATEGORIES),
        "hour": pd.Timestamp(txn["timestamp"]).hour,
    }

def _rule_score(f):
    s = 0.0
    if f["amount_ratio"] >= 5:   s += 0.4
    elif f["amount_ratio"] >= 3: s += 0.2
    if f["risky_category"]:      s += 0.3
    if f["new_payee"]:           s += 0.2
    if f["hour"] < 6 or f["hour"] >= 22: s += 0.1
    return min(s, 1.0)

def _model_score(f):
    if _model is None:
        return 0.0
    X = pd.DataFrame([f])[_model["features"]]
    a = -_model["model"].score_samples(X)[0]
    return float(np.clip((a - _model["p50"]) / (_model["p99"] - _model["p50"]), 0, 1))

def score_transaction(txn):
    _load()
    f = _features(txn)
    score = max(_model_score(f), _rule_score(f))
    reasons = []
    if f["amount_ratio"] >= 3:   reasons.append(f"{f['amount_ratio']:.0f}x usual amount")
    if f["risky_category"]:      reasons.append(RISKY_REASON)
    if f["new_payee"]:           reasons.append("new payee")
    if f["hour"] < 6 or f["hour"] >= 22: reasons.append("unusual hour")
    return {"score": round(score, 3), "reasons": reasons}

Overwriting /home/ibm-user/shared/safecircle/src/transactions.py


In [1]:
import sys, pandas as pd
REPO = "/home/ibm-user/shared/safecircle"
sys.path.insert(0, REPO)
from src.transactions import score_transaction

txns = pd.read_csv(f"{REPO}/scenarios/transactions.csv")
for t in txns.to_dict("records"):
    print(t["user_id"], score_transaction(t))

FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/shared/safecircle/scenarios/history.csv'

In [2]:
%%writefile /home/ibm-user/shared/safecircle/src/transactions.py
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

REPO = Path(__file__).absolute().parent.parent
MODEL_PATH = REPO / "models" / "transaction_model.joblib"
HISTORY_PATH = REPO / "scenarios" / "history.csv"

RISKY_CATEGORIES = {"gift_card", "wire", "money_transfer"}
RISKY_REASON = "gift-card or wire payment"   # risk.py looks for this exact text

_model = None
_history = None

def _load():
    global _model, _history
    if _history is None:
        _history = pd.read_csv(HISTORY_PATH)
    if _model is None and MODEL_PATH.exists():
        _model = joblib.load(MODEL_PATH)

def _features(txn):
    amount = float(txn["amount"])
    past = _history[_history["user_id"] == txn["user_id"]]
    usual = past["amount"].median() if len(past) else 50.0
    return {
        "amount_ratio": min(amount / usual, 100.0) if usual > 0 else 1.0,
        "new_payee": int(txn.get("is_new_payee", 0)),
        "risky_category": int(str(txn.get("merchant_category", "")).lower() in RISKY_CATEGORIES),
        "hour": pd.Timestamp(txn["timestamp"]).hour,
    }

def _rule_score(f):
    s = 0.0
    if f["amount_ratio"] >= 5:   s += 0.4
    elif f["amount_ratio"] >= 3: s += 0.2
    if f["risky_category"]:      s += 0.3
    if f["new_payee"]:           s += 0.2
    if f["hour"] < 6 or f["hour"] >= 22: s += 0.1
    return min(s, 1.0)

def _model_score(f):
    if _model is None:
        return 0.0
    X = pd.DataFrame([f])[_model["features"]]
    a = -_model["model"].score_samples(X)[0]
    return float(np.clip((a - _model["p50"]) / (_model["p99"] - _model["p50"]), 0, 1))

def score_transaction(txn):
    _load()
    f = _features(txn)
    score = max(_model_score(f), _rule_score(f))
    reasons = []
    if f["amount_ratio"] >= 3:   reasons.append(f"{f['amount_ratio']:.0f}x usual amount")
    if f["risky_category"]:      reasons.append(RISKY_REASON)
    if f["new_payee"]:           reasons.append("new payee")
    if f["hour"] < 6 or f["hour"] >= 22: reasons.append("unusual hour")
    return {"score": round(score, 3), "reasons": reasons}

Overwriting /home/ibm-user/shared/safecircle/src/transactions.py


In [3]:
import sys, pandas as pd
REPO = "/home/ibm-user/shared/safecircle"
sys.path.insert(0, REPO)
from src.transactions import score_transaction

txns = pd.read_csv(f"{REPO}/scenarios/transactions.csv")
for t in txns.to_dict("records"):
    print(t["user_id"], score_transaction(t))

rosa {'score': 1.0, 'reasons': ['43x usual amount', 'gift-card or wire payment', 'new payee']}
walter {'score': 0.0, 'reasons': []}
linda {'score': 0.917, 'reasons': ['5x usual amount', 'gift-card or wire payment', 'new payee']}


In [2]:
import os, pandas as pd, numpy as np
HIST = "/home/ibm-user/shared/safecircle/scenarios/history.csv"
os.makedirs(os.path.dirname(HIST), exist_ok=True)

rng = np.random.default_rng(42)
people = {
    "rosa":   [("Kroger", "grocery", (30, 90)), ("Walgreens", "pharmacy", (10, 40)), ("Corner Cafe", "restaurant", (8, 20))],
    "walter": [("CVS Pharmacy", "pharmacy", (15, 60)), ("Tom Thumb", "grocery", (25, 80)), ("Shell", "gas", (30, 55))],
    "linda":  [("Target", "retail", (20, 120)), ("Kroger", "grocery", (40, 110)), ("Atmos Energy", "utilities", (60, 95))],
}
rows = []
for user, stores in people.items():
    for i in range(12):
        merchant, cat, (lo, hi) = stores[i % len(stores)]
        when = pd.Timestamp("2026-09-01") + pd.Timedelta(days=3 * i, hours=int(rng.integers(9, 18)))
        rows.append({"user_id": user, "timestamp": when.strftime("%Y-%m-%d %H:%M"),
                     "amount": round(float(rng.uniform(lo, hi)), 2),
                     "merchant": merchant, "merchant_category": cat})

pd.DataFrame(rows).to_csv(HIST, index=False)
print("exists now:", os.path.exists(HIST))
print(os.listdir(os.path.dirname(HIST)))

exists now: True
['transactions.csv', '.ipynb_checkpoints', 'messages.csv', 'history.csv']


In [4]:
%%writefile /home/ibm-user/shared/safecircle/src/risk.py
HOLD_THRESHOLD = 0.6      # tune this in Phase 3
SAFETY_MSG_SCORE = 0.6
RISKY_REASON = "gift-card or wire payment"   # must match src/transactions.py

def decide(msg_result, txn_result):
    msg = float(msg_result["score"])
    txn = float(txn_result["score"])
    risk = 0.5 * msg + 0.5 * txn

    # Safety rule: scam-like message + gift card/wire payment = always hold
    safety_rule = msg >= SAFETY_MSG_SCORE and RISKY_REASON in txn_result["reasons"]
    if safety_rule:
        risk = max(risk, HOLD_THRESHOLD)

    action = "hold_and_alert" if risk >= HOLD_THRESHOLD else "allow"
    reasons = list(dict.fromkeys(msg_result["reasons"] + txn_result["reasons"]))
    if safety_rule:
        reasons.append("scam-like message just before a gift-card or wire payment")
    return {"risk": round(risk, 3), "action": action, "reasons": reasons}

Overwriting /home/ibm-user/shared/safecircle/src/risk.py


In [1]:
import sys, pandas as pd
REPO = "/home/ibm-user/shared/safecircle"
sys.path.insert(0, REPO)
from src.transactions import score_transaction
from src.risk import decide

fake_msg = {"rosa":   {"score": 0.9, "reasons": ["asks for gift cards"]},
            "walter": {"score": 0.1, "reasons": []},
            "linda":  {"score": 0.7, "reasons": ["account locked"]}}

txns = pd.read_csv(f"{REPO}/scenarios/transactions.csv")
for t in txns.to_dict("records"):
    d = decide(fake_msg[t["user_id"]], score_transaction(t))
    ok = d["action"] == t["expected_action"]
    print(f"{t['user_id']:7} {d['action']:15} expected {t['expected_action']:15} "
          f"{'PASS' if ok else 'FAIL'}  risk={d['risk']}")

rosa    hold_and_alert  expected hold_and_alert  PASS  risk=0.95
walter  allow           expected allow           PASS  risk=0.05
linda   hold_and_alert  expected hold_and_alert  PASS  risk=0.808


In [2]:
TRUSTED = {"rosa": "her daughter", "walter": "his son", "linda": "her sister"}

def family_alert(txn, decision):
    who = txn["user_id"].capitalize()
    what = str(txn["merchant_category"]).replace("_", "-")
    return (f"To {who}'s trusted contact ({TRUSTED.get(txn['user_id'], 'family')}): "
            f"SafeCircle paused a ${float(txn['amount']):,.0f} {what} payment on {who}'s card "
            f"after a likely scam message ({', '.join(decision['reasons'][:3])}). Please call {who}.")

for t in txns.to_dict("records"):
    d = decide(fake_msg[t["user_id"]], score_transaction(t))
    print(f"{t['user_id']:7} {d['action']:15} risk={d['risk']}")
    if d["action"] == "hold_and_alert":
        print("   FAMILY ALERT:", family_alert(t, d))
    print()

rosa    hold_and_alert  risk=0.95
   FAMILY ALERT: To Rosa's trusted contact (her daughter): SafeCircle paused a $1,500 gift-card payment on Rosa's card after a likely scam message (asks for gift cards, 43x usual amount, gift-card or wire payment). Please call Rosa.

walter  allow           risk=0.05

linda   hold_and_alert  risk=0.808
   FAMILY ALERT: To Linda's trusted contact (her sister): SafeCircle paused a $300 wire payment on Linda's card after a likely scam message (account locked, 5x usual amount, gift-card or wire payment). Please call Linda.



In [3]:
import os
print(os.listdir("/home/ibm-user/shared"))

['Untitled.ipynb', 'safecircle', 'safecircle', '.Trash-1000', 'Untitled2.ipynb', 'Untitled1.ipynb', '.ipynb_checkpoints', 'staging', 'detector.ipynb', 'authors.txt']


In [4]:
import os
REPO = "/home/ibm-user/shared/safecircle"
for folder in ["data", "scenarios", "src", "models", "notebooks/pairA", "notebooks/pairB", "docs/screenshots"]:
    os.makedirs(f"{REPO}/{folder}", exist_ok=True)

with open(f"{REPO}/.gitignore", "a") as f:
    f.write("data/\n.ipynb_checkpoints/\n__pycache__/\n")
print(os.listdir(REPO))

['docs', '.gitignore', 'notebooks', 'scenarios', 'data', 'src', 'models', '.ipynb_checkpoints', 'README.md', '.git']
